In [5]:
import sys
import os

# Add parent directory to path to import from app
sys.path.insert(0, os.path.abspath('..'))

from app.db.core.models.market_data_models import *
from app.db.core.models.user_data_models import *
from app.db.core.models.prophit_alts_models import *
from app.db.core.db_config import UserSession, MarketSession, ProphitAltsSession
from app.utils.decorators.database import with_session, with_transaction
from app.utils.serialize_output import serialize_sqlalchemy_obj
import json
from app.utils.gpt_parser import canonical_portfolio

In [6]:
portfolio = {'AAPL': 0.1, 'SPY': 0.1, 'TSLA': 0.1, 'NVDA': 0.1, 'MSFT': 0.1, 'GOOG': 0.1, 'AMZN': 0.1, 'NFLX': 0.1}
portfolio = canonical_portfolio(portfolio)
print(portfolio)

{'AAPL': {'allocation': 0.1, 'position': 'long'}, 'SPY': {'allocation': 0.1, 'position': 'long'}, 'TSLA': {'allocation': 0.1, 'position': 'long'}, 'NVDA': {'allocation': 0.1, 'position': 'long'}, 'MSFT': {'allocation': 0.1, 'position': 'long'}, 'GOOG': {'allocation': 0.1, 'position': 'long'}, 'AMZN': {'allocation': 0.1, 'position': 'long'}, 'NFLX': {'allocation': 0.1, 'position': 'long'}}


In [7]:
# Import existing risk calculators
from app.core.calculations.risk.calculator import RiskCalculator
from app.core.calculations.risk.liquidity import LiquidityCalculator
from app.repositories.price_data import get_price_data_daily
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

print("Risk calculation modules loaded!")


Risk calculation modules loaded!


In [8]:
def calculate_market_risk(ticker, lookback_days=252, market_ticker='SPY'):
    """
    Calculate comprehensive market risk metrics for a ticker.
    
    Metrics calculated:
    - Annualized Volatility
    - Historical VaR (95% and 99%)
    - Expected Shortfall (CVaR)
    - Beta to market
    - Max Drawdown
    - Ulcer Index
    
    Args:
        ticker: Stock ticker symbol
        lookback_days: Days of historical data (default 252 = 1 year)
        market_ticker: Market benchmark ticker (default 'SPY')
    
    Returns:
        Dict with market risk metrics and normalized score (0-1)
    """
    print(f"Calculating market risk for {ticker}...")
    
    # Fetch price data
    end_date = datetime.now()
    start_date = end_date - timedelta(days=lookback_days + 50)  # Extra buffer
    
    price_data = get_price_data_daily(ticker, start_date, end_date)
    market_data = get_price_data_daily(market_ticker, start_date, end_date)
    
    if price_data.empty:
        return {'error': f'No price data available for {ticker}'}
    
    # Calculate daily returns
    daily_returns = price_data['close'].pct_change().dropna()
    
    # 1. Annualized Volatility
    volatility = RiskCalculator.annualized_volatility(daily_returns)
    
    # 2. Historical VaR (95% and 99%)
    var_95 = RiskCalculator.historical_var(daily_returns, confidence=0.95)
    var_99 = RiskCalculator.historical_var(daily_returns, confidence=0.99)
    
    # 3. Expected Shortfall (CVaR)
    cvar_95 = RiskCalculator.expected_shortfall(daily_returns, confidence=0.95)
    cvar_99 = RiskCalculator.expected_shortfall(daily_returns, confidence=0.99)
    
    # 4. Beta to market
    if not market_data.empty:
        market_returns = market_data['close'].pct_change().dropna()
        beta = RiskCalculator.beta(daily_returns, market_returns)
        up_beta, down_beta = RiskCalculator.up_down_beta(daily_returns, market_returns)
    else:
        beta = np.nan
        up_beta, down_beta = np.nan, np.nan
    
    # 5. Max Drawdown
    max_dd = RiskCalculator.max_drawdown(price_data['close'])
    
    # 6. Ulcer Index (downside volatility)
    ulcer = RiskCalculator.ulcer_index(price_data['close'])
    
    # Normalize to 0-1 risk score (higher = more risky)
    # Using volatility as primary indicator
    # Market average vol ~15-20%, high vol stocks >30%
    if not np.isnan(volatility):
        if volatility < 0.15:
            vol_score = 0.3  # Low risk
        elif volatility < 0.25:
            vol_score = 0.5  # Moderate risk
        elif volatility < 0.40:
            vol_score = 0.7  # High risk
        else:
            vol_score = min(0.9, 0.7 + (volatility - 0.40) * 0.5)  # Very high risk
    else:
        vol_score = 0.5
    
    # Adjust by drawdown severity
    if not np.isnan(max_dd):
        dd_penalty = min(abs(max_dd) * 0.5, 0.3)  # Up to 0.3 penalty for severe drawdowns
        vol_score = min(vol_score + dd_penalty, 1.0)
    
    # Assign grade
    if vol_score < 0.30:
        grade = 'A'
    elif vol_score < 0.50:
        grade = 'B'
    elif vol_score < 0.70:
        grade = 'C'
    elif vol_score < 0.85:
        grade = 'D'
    else:
        grade = 'F'
    
    return {
        'ticker': ticker,
        'market_risk_score': vol_score,
        'grade': grade,
        'metrics': {
            'annualized_volatility': volatility,
            'var_95': var_95,
            'var_99': var_99,
            'cvar_95': cvar_95,
            'cvar_99': cvar_99,
            'beta': beta,
            'up_beta': up_beta,
            'down_beta': down_beta,
            'max_drawdown': max_dd,
            'ulcer_index': ulcer
        }
    }

print("Market risk calculation function defined!")


Market risk calculation function defined!


In [9]:
# Test with a single ticker - AAPL
aapl_risk = calculate_market_risk('AAPL')

print(f"\n{'='*60}")
print(f"MARKET RISK ANALYSIS: {aapl_risk['ticker']}")
print(f"{'='*60}")
print(f"Overall Score: {aapl_risk['market_risk_score']:.3f} | Grade: {aapl_risk['grade']}")
print(f"\nDetailed Metrics:")
print(f"  Annualized Volatility: {aapl_risk['metrics']['annualized_volatility']:.2%}")
print(f"  VaR (95%): {aapl_risk['metrics']['var_95']:.2%}")
print(f"  VaR (99%): {aapl_risk['metrics']['var_99']:.2%}")
print(f"  CVaR (95%): {aapl_risk['metrics']['cvar_95']:.2%}")
print(f"  CVaR (99%): {aapl_risk['metrics']['cvar_99']:.2%}")
print(f"  Beta (vs SPY): {aapl_risk['metrics']['beta']:.2f}")
print(f"  Up Beta: {aapl_risk['metrics']['up_beta']:.2f}")
print(f"  Down Beta: {aapl_risk['metrics']['down_beta']:.2f}")
print(f"  Max Drawdown: {aapl_risk['metrics']['max_drawdown']:.2%}")
print(f"  Ulcer Index: {aapl_risk['metrics']['ulcer_index']:.4f}")
print(f"{'='*60}")


Calculating market risk for AAPL...

MARKET RISK ANALYSIS: AAPL
Overall Score: 0.866 | Grade: F

Detailed Metrics:
  Annualized Volatility: 34.37%
  VaR (95%): 3.27%
  VaR (99%): 5.21%
  CVaR (95%): 4.70%
  CVaR (99%): 7.29%
  Beta (vs SPY): 1.30
  Up Beta: 1.40
  Down Beta: 1.39
  Max Drawdown: -33.27%
  Ulcer Index: 0.1588


In [10]:
# Calculate market risk for entire portfolio
print("Calculating market risk for entire portfolio...\n")

portfolio_market_risk = {}
for ticker, details in portfolio.items():
    try:
        risk_result = calculate_market_risk(ticker)
        if 'error' not in risk_result:
            portfolio_market_risk[ticker] = risk_result
            print(f"{ticker:6} | Score: {risk_result['market_risk_score']:.3f} | Grade: {risk_result['grade']} | Vol: {risk_result['metrics']['annualized_volatility']:.1%}")
    except Exception as e:
        print(f"{ticker:6} | ERROR: {str(e)}")

print(f"\nCompleted analysis for {len(portfolio_market_risk)}/{len(portfolio)} tickers")


Calculating market risk for entire portfolio...

Calculating market risk for AAPL...
AAPL   | Score: 0.866 | Grade: F | Vol: 34.4%
Calculating market risk for SPY...
SPY    | Score: 0.595 | Grade: C | Vol: 20.0%
Calculating market risk for TSLA...
TSLA   | Score: 1.000 | Grade: F | Vol: 69.4%
Calculating market risk for NVDA...
NVDA   | Score: 0.940 | Grade: F | Vol: 50.8%
Calculating market risk for MSFT...
MSFT   | Score: 0.603 | Grade: C | Vol: 23.7%
Calculating market risk for GOOG...
Calculating market risk for AMZN...
AMZN   | Score: 0.849 | Grade: D | Vol: 33.9%
Calculating market risk for NFLX...
NFLX   | Score: 0.797 | Grade: D | Vol: 34.3%

Completed analysis for 7/8 tickers


In [11]:
# Calculate portfolio-level market risk (weighted average)
print("\n" + "="*60)
print("PORTFOLIO-LEVEL MARKET RISK")
print("="*60)

total_weight = sum([portfolio[t]['allocation'] for t in portfolio_market_risk.keys()])

# Weighted average risk score
weighted_risk = sum([
    portfolio_market_risk[t]['market_risk_score'] * portfolio[t]['allocation'] 
    for t in portfolio_market_risk.keys()
]) / total_weight if total_weight > 0 else 0

# Weighted average volatility
weighted_vol = sum([
    portfolio_market_risk[t]['metrics']['annualized_volatility'] * portfolio[t]['allocation'] 
    for t in portfolio_market_risk.keys()
]) / total_weight if total_weight > 0 else 0

# Weighted average beta
weighted_beta = sum([
    portfolio_market_risk[t]['metrics']['beta'] * portfolio[t]['allocation'] 
    for t in portfolio_market_risk.keys()
]) / total_weight if total_weight > 0 else 0

# Portfolio grade
if weighted_risk < 0.30:
    portfolio_grade = 'A'
elif weighted_risk < 0.50:
    portfolio_grade = 'B'
elif weighted_risk < 0.70:
    portfolio_grade = 'C'
elif weighted_risk < 0.85:
    portfolio_grade = 'D'
else:
    portfolio_grade = 'F'

print(f"\nWeighted Portfolio Metrics:")
print(f"  Overall Risk Score: {weighted_risk:.3f} | Grade: {portfolio_grade}")
print(f"  Average Volatility: {weighted_vol:.2%}")
print(f"  Portfolio Beta: {weighted_beta:.2f}")
print(f"  Number of Holdings: {len(portfolio_market_risk)}")

# Risk distribution
print(f"\nRisk Grade Distribution:")
grade_counts = {}
for ticker, risk_data in portfolio_market_risk.items():
    grade = risk_data['grade']
    grade_counts[grade] = grade_counts.get(grade, 0) + 1

for grade in ['A', 'B', 'C', 'D', 'F']:
    count = grade_counts.get(grade, 0)
    pct = (count / len(portfolio_market_risk) * 100) if len(portfolio_market_risk) > 0 else 0
    bar = '█' * int(pct / 5)
    print(f"  Grade {grade}: {count:2} stocks ({pct:5.1f}%) {bar}")

print("="*60)



PORTFOLIO-LEVEL MARKET RISK

Weighted Portfolio Metrics:
  Overall Risk Score: 0.807 | Grade: D
  Average Volatility: 38.06%
  Portfolio Beta: 1.39
  Number of Holdings: 7

Risk Grade Distribution:
  Grade A:  0 stocks (  0.0%) 
  Grade B:  0 stocks (  0.0%) 
  Grade C:  2 stocks ( 28.6%) █████
  Grade D:  2 stocks ( 28.6%) █████
  Grade F:  3 stocks ( 42.9%) ████████


In [12]:
# Create summary DataFrame for easy analysis
summary_data = []
for ticker, risk_data in portfolio_market_risk.items():
    summary_data.append({
        'Ticker': ticker,
        'Allocation': portfolio[ticker]['allocation'],
        'Risk Score': risk_data['market_risk_score'],
        'Grade': risk_data['grade'],
        'Volatility': risk_data['metrics']['annualized_volatility'],
        'Beta': risk_data['metrics']['beta'],
        'VaR 95%': risk_data['metrics']['var_95'],
        'Max Drawdown': risk_data['metrics']['max_drawdown']
    })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values('Risk Score', ascending=False)

print("\nPORTFOLIO MARKET RISK SUMMARY")
print("="*100)
print(df_summary.to_string(index=False))
print("="*100)



PORTFOLIO MARKET RISK SUMMARY
Ticker  Allocation  Risk Score Grade  Volatility     Beta  VaR 95%  Max Drawdown
  TSLA         0.1    1.000000     F    0.693520 2.347851 0.060409     -0.519983
  NVDA         0.1    0.939886     F    0.507945 1.903441 0.048757     -0.371827
  AAPL         0.1    0.866352     F    0.343665 1.300440 0.032699     -0.332703
  AMZN         0.1    0.849487     D    0.339480 1.345187 0.031654     -0.298973
  NFLX         0.1    0.797346     D    0.342649 0.938026 0.028862     -0.194691
  MSFT         0.1    0.603088     C    0.236942 0.894906 0.019791     -0.206177
   SPY         0.1    0.594600     C    0.199901 1.000000 0.016121     -0.189200


---

## Market Risk Metrics Explained

### Risk Score (0-1 scale)
- **0.0-0.3 (Grade A)**: Very Low Risk - Stable, defensive stocks
- **0.3-0.5 (Grade B)**: Low Risk - Market-average volatility
- **0.5-0.7 (Grade C)**: Moderate Risk - Somewhat volatile
- **0.7-0.85 (Grade D)**: High Risk - Highly volatile
- **0.85-1.0 (Grade F)**: Very High Risk - Extremely volatile

### Key Metrics

**Annualized Volatility**: Standard deviation of returns (annualized)
- Low: <15% | Moderate: 15-25% | High: 25-40% | Very High: >40%

**VaR (Value at Risk)**: Maximum expected loss at confidence level
- VaR 95% means: 95% chance losses won't exceed this amount in 1 day
- VaR 99% means: 99% chance losses won't exceed this amount in 1 day

**CVaR (Expected Shortfall)**: Average loss when losses exceed VaR
- More comprehensive than VaR (captures tail risk)

**Beta**: Sensitivity to market movements
- Beta = 1.0: Moves with market
- Beta > 1.0: More volatile than market (amplifies moves)
- Beta < 1.0: Less volatile than market (defensive)
- Up Beta vs Down Beta: Shows asymmetric behavior in bull/bear markets

**Max Drawdown**: Largest peak-to-trough decline
- Shows worst historical loss from any high point

**Ulcer Index**: RMS of drawdowns (downside volatility measure)
- Focuses on downside risk rather than total volatility
